## Visual Autoencoder pretraining

In [ ]:
!pip install torchinfo
!pip install clip
!pip install evaluate
!pip install rouge_score
!pip install nltk

  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-0.2.0-py3-none-any.whl size=6989 sha256=080cdb97e2b0e08560564caaf793e74a7b0fe6926a30125c88c8b00d247d236e
  Stored in directory: /root/.cache/pip/wheels/6c/fd/54/9d4e15cf829b871199a7cd3597e869a514d1624a0a43076896
Successfully built clip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=dc6afb7eb4c96394b95f9f2424a57e29f2772c072d88952a94897cbf6be8643f
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
# @title Importing the necessary libraries

import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
from bs4 import BeautifulSoup
import re
import matplotlib.pyplot as plt
import numpy as np
import os
from nltk.translate.bleu_score import sentence_bleu
import json
import pandas as pd
from torchinfo import summary
from transformers import CLIPProcessor, CLIPModel, RobertaModel, RobertaTokenizer
import evaluate
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import tqdm
from datasets.fingerprint import random
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms.functional as FT
import math
from transformers import BertTokenizer
import gc

import textwrap

# **Chapter 1: The data preparation**


---



First we need to activate our google drive so that we can save out data permanently.

## 1.1 Loading and saving data

We need to define a couple of functions to make our life easier. Feel free to tweak those functions:

In [ ]:

emb_dim = 128
latent_dim = 128
num_layers = 6
max_seq_len = 120
batch_size = 16
dropout = 0.1

Now we load dataset from HuggingFace:

In [ ]:
# @title Loading the dataset
from datasets import load_dataset

train_dataset = load_dataset("daniel3303/StoryReasoning", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.94k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/327M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3552 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/626 [00:00<?, ? examples/s]

## 1.2 Three datasets


---



We will create three different dataset objects and the corresponding loaders for performing multiple tasks

In [ ]:
# @title CoT improvements toggles (added)
"""
Configuration flags to control which Chain-of-Thought (CoT) mechanisms are used during training:
- `USE_FRAME_AWARE_GROUNDING`: Aligns ROI embeddings with the text embedding of the specific frame (vs global).
- `USE_CONTRASTIVE_ROI`: Enables InfoNCE loss to contrast positive ROI-text pairs against negatives.
- `USE_ENTITY_POOLING`: Enforces consistency of embeddings for the same entity within a batch.
- `USE_COT_TEXT`: Appends CoT reasoning text to the input frame description.
"""

# Turn these on/off to control the 4 optional improvements.
USE_FRAME_AWARE_GROUNDING = True      # Option 2: align ROI to matching frame text embedding (instead of always frame 0)
USE_CONTRASTIVE_ROI = True            # Option 1: InfoNCE-style contrastive grounding using batch negatives
USE_ENTITY_POOLING = True             # Option 3: entity-specific pooling/consistency across batch by entity_id
USE_COT_TEXT = True                   # Option 4: concatenate CoT text snippet to the frame descriptions

# Contrastive temperature (only used if USE_CONTRASTIVE_ROI=True)
CONTRASTIVE_TAU = 0.07

In [ ]:
# @title Dataset for image autoencoder task
"""
Defines `AutoEncoderTaskDataset` for pre-training the visual autoencoder.
It retrieves a single random frame from the dataset to learn image reconstruction (Image -> Latent -> Image).
"""

class AutoEncoderTaskDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        self.transform = transforms.Compose([
          transforms.Resize((224, 224)),# Reasonable size based on our previous analysis
          transforms.ToTensor(), # HxWxC -> CxHxW
        ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
      num_frames = self.dataset[idx]["frame_count"]
      frames = self.dataset[idx]["images"]

      # Pick a frame at random
      frame_idx = torch.randint(0, 5, (1,)).item()
      input_frame = self.transform(frames[frame_idx]) # Input to the autoencoder

      return input_frame, # Returning the image

## 1.3 Creating and testing our dataset objects and loaders


---



In [ ]:
print(train_dataset[0].keys())

dict_keys(['story_id', 'images', 'frame_count', 'chain_of_thought', 'story'])


In [ ]:
# @title For the image autoencoder task
"""
Sets up the data pipeline for the auxiliary visual autoencoding task.
Creates the `AutoEncoderTaskDataset` and its `DataLoader`.
"""

autoencoder_dataset = AutoEncoderTaskDataset(train_dataset)
autoencoder_dataloader = DataLoader(autoencoder_dataset, batch_size=batch_size, shuffle=True)

# **Models**


---



## The Vision models

In [ ]:
# =========================================================
# Predictive Visual Latent Autoencoder
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel
from torchvision import models

# =========================================================
# Residual Block
# =========================================================

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.GroupNorm(8, channels)
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


# =========================================================
# Attention Block
# =========================================================

class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=8):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.attn = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).view(B, C, H * W).transpose(1, 2)
        attn_out, _ = self.attn(h, h, h)
        return x + attn_out.transpose(1, 2).view(B, C, H, W)


# =========================================================
# CLIP Encoder (Predictive + Stochastic)
# =========================================================

class CLIPEncoderWrapper(nn.Module):
    def __init__(self, latent_dim=256, spatial_dim=256, unfreeze_layers=4):
        super().__init__()

        self.clip = CLIPModel.from_pretrained(
            "openai/clip-vit-base-patch16"
        )

        self.clip.vision_model.config.output_hidden_states = True

        for p in self.clip.parameters():
            p.requires_grad = False

        for layer in self.clip.vision_model.encoder.layers[-unfreeze_layers:]:
            for p in layer.parameters():
                p.requires_grad = True

        hidden = self.clip.config.vision_config.hidden_size

        # -------- VAE heads --------
        self.mu = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Linear(512, latent_dim)
        )

        self.logvar = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.GELU(),
            nn.Linear(512, latent_dim)
        )

        # -------- spatial latent --------
        self.spatial = nn.Sequential(
            nn.Conv2d(768, 512, 1),
            nn.GroupNorm(8, 512),
            nn.GELU(),
            nn.Conv2d(512, spatial_dim, 1)
        )

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):

        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)

        mean = torch.tensor([0.481, 0.457, 0.408], device=x.device).view(1,3,1,1)
        std  = torch.tensor([0.268, 0.261, 0.275], device=x.device).view(1,3,1,1)
        x = (x - mean) / std

        out = self.clip.vision_model(pixel_values=x, return_dict=True, output_hidden_states=True)

        pooled = out.pooler_output

        mu = self.mu(pooled)
        logvar = self.logvar(pooled)
        z = self.reparam(mu, logvar)

        # Ensure hidden_states exist (needed for torchinfo)
        h_states = out.hidden_states if out.hidden_states is not None else [out.last_hidden_state]
        patches = h_states[-1][:, 1:, :].transpose(1, 2)
        B = patches.shape[0]
        patches = patches.reshape(B, 768, 14, 14)

        spatial_z = self.spatial(patches)

        return z, mu, logvar, spatial_z


# =========================================================
# Decoder
# =========================================================

class VisualDecoder(nn.Module):
    def __init__(self, latent_dim=256, spatial_dim=256):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 256 * 14 * 14)

        self.fuse = nn.Conv2d(256 + spatial_dim, 256, 3, padding=1)

        self.up = nn.Sequential(
            ResidualBlock(256),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.GELU(),
            ResidualBlock(128),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.GELU(),
            ResidualBlock(64),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.GELU(),
            ResidualBlock(32),

            nn.ConvTranspose2d(32, 16, 4, 2, 1),
            nn.GELU(),
            ResidualBlock(16),

            nn.Conv2d(16, 3, 3, padding=1),
            nn.Tanh()
        )

    def forward(self, z, spatial_z):

        B = z.size(0)

        x = self.fc(z).view(B, 256, 14, 14)

        x = torch.cat([x, spatial_z], dim=1)

        x = self.fuse(x)

        return self.up(x)


# =========================================================
# FULL AUTOENCODER (NOW PREDICTIVE)
# =========================================================

class VisualAutoencoder(nn.Module):
    def __init__(self, latent_dim=256, spatial_dim=256):
        super().__init__()

        self.encoder = CLIPEncoderWrapper(latent_dim, spatial_dim)
        self.decoder = VisualDecoder(latent_dim, spatial_dim)

        # -------- CRITICAL ADDITION --------
        # latent predictor (temporal modeling)
        self.latent_predictor = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.GELU(),
            nn.Linear(latent_dim, latent_dim)
        )

    def forward(self, x):

        z, mu, logvar, spatial = self.encoder(x)

        x_hat = self.decoder(z, spatial)

        return x_hat, z, mu, logvar, spatial

    # predict next latent (KEY FOR SEQUENCE MODEL)
    def predict_next_latent(self, z_t):
        return self.latent_predictor(z_t)


# =========================================================
# LOSSES
# =========================================================

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_FEATURES)
        self.features = nn.Sequential(*list(vgg.features[:16])).eval()
        for p in self.features.parameters():
            p.requires_grad = False

    def forward(self, pred, target):
        return F.l1_loss(self.features(pred), self.features(target))


class KLLoss(nn.Module):
    def forward(self, mu, logvar):
        return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())


class ReconstructionLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.perc = PerceptualLoss()
        self.kl = KLLoss()

    def forward(self, pred, target, mu, logvar):

        pixel = 0.5 * F.mse_loss(pred, target) + 0.5 * F.l1_loss(pred, target)
        perceptual = self.perc(pred, target)
        kl = self.kl(mu, logvar)

        return {
            "total": pixel + 0.3 * perceptual + 0.01 * kl,
            "pixel": pixel,
            "perceptual": perceptual,
            "kl": kl
        }

## 2.3 The main architecture


### Implementation Note
By using `MultiHeadTemporalAttention`, we ensure that the `SequencePredictor` doesn't just rely on the final GRU hidden state (which can lose information over long sequences), but instead learns which specific frames in the context are most relevant for denoising the target frame.

# **Training routines**


---




In [ ]:
def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
        # Use a slightly higher gain for leaky_relu to boost signal
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.01) # Small positive bias to keep neurons alive
    elif isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0)

## 3.1 Initialization and setup

In [ ]:
# @title Initializing visual models
"""
1. Instantiates the `VisualAutoencoder`.
2. Applies the custom weight initialization.
"""

visual_autoencoder = VisualAutoencoder(latent_dim=256).to(device)
visual_autoencoder.decoder.apply(init_weights)
#visual_autoencoder, _, _, _ = load_checkpoint_from_drive(visual_autoencoder, None, filename='visual_autoencoder.pth')
total_params = sum(p.numel() for p in visual_autoencoder.parameters() if p.requires_grad)
print(f"Total trainable parameters in visual autoencoder: {total_params}")

config.json:   0%|          | 0.00/4.10k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Checkpoint loaded from: /content/drive/MyDrive/DL_Checkpoints/visual_autoencoder.pth (epoch 14)
Total trainable parameters in visual autoencoder: 46405699


In [ ]:
# 3-channel RGB image, 64x64 resolution
dmy_image = torch.randn(batch_size, 3, 224, 224).to(device)

print("===== CLIP Visual Autoencoder Summary =====")
summary(
    visual_autoencoder,
    input_data=dmy_image,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    depth=3
)

===== CLIP Visual Autoencoder Summary =====


Layer (type:depth-idx)                                                 Input Shape               Output Shape              Param #                   Trainable
VisualAutoencoder                                                      [16, 3, 224, 224]         [16, 3, 224, 224]         131,584                   Partial
├─CLIPEncoderWrapper: 1-1                                              [16, 3, 224, 224]         [16, 256]                 --                        Partial
│    └─CLIPModel: 2-1                                                  --                        --                        63,821,313                Partial
│    │    └─CLIPVisionTransformer: 3-1                                 --                        [16, 768]                 85,799,424                Partial
│    └─Sequential: 2-2                                                 [16, 768]                 [16, 256]                 --                        True
│    │    └─Linear: 3-2                                    

## 3.2 Training loops

In [ ]:
def sequence_readiness_test_v3(
    model,
    dataloader,
    device,
    steps=1500,
    lr=1e-4,
    noise_std=0.02,
    visualize_every=200
):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # Handle tuple return from dataloader
    batch = next(iter(dataloader))
    if isinstance(batch, (list, tuple)):
        frames = batch[0][:1].to(device)
    else:
        frames = batch[:1].to(device)

    frames = F.interpolate(frames, (224, 224))

    print("\n===== SEQUENCE READINESS TRAINING TEST =====\n")

    for step in range(steps):
        optimizer.zero_grad()

        x_t = frames + torch.randn_like(frames) * noise_std
        x_t1 = frames + torch.randn_like(frames) * noise_std

        # Unpack the 5 values returned by VisualAutoencoder
        z_t, mu_t, log_t, spatial_t = model.encoder(x_t)
        z_t1, mu_t1, log_t1, spatial_t1 = model.encoder(x_t1)

        # Get reconstruction from full forward pass
        recon, _, _, _, _ = model(x_t)

        recon_loss = (
            0.5 * F.mse_loss(recon, frames) +
            0.5 * F.l1_loss(recon, frames)
        )

        latent_consistency = F.mse_loss(z_t, z_t1)

        # Use spatial consistency
        spatial_consistency = F.mse_loss(spatial_t, spatial_t1)

        loss = recon_loss + 0.2 * latent_consistency + 0.1 * spatial_consistency

        loss.backward()
        optimizer.step()

        if step % visualize_every == 0 or step == steps - 1:
            model.eval()
            with torch.no_grad():
                z0, _, _, _ = model.encoder(x_t)
                z1, _, _, _ = model.encoder(x_t1)
                latent_drift = F.mse_loss(z0, z1).item()

                recon_eval, _, _, _, _ = model(x_t)
                recon_error = F.mse_loss(recon_eval, frames).item()

                print(f"Step {step} | Recon: {recon_error:.5f} | Latent drift: {latent_drift:.5f}")

                fig, ax = plt.subplots(1, 2, figsize=(8,4))
                ax[0].imshow(frames[0].permute(1,2,0).cpu().clamp(0,1))
                ax[0].set_title("Target")
                ax[1].imshow(recon_eval[0].permute(1,2,0).cpu().clamp(0,1))
                ax[1].set_title(f"Recon Step {step}")
                plt.show()
            model.train()

    print("\n===== TEST COMPLETE === stone")

In [ ]:
# =========================================================
# VISUAL AUTOENCODER PRETRAINING
# =========================================================

from tqdm import tqdm

start_epoch_v = 0
N_EPOCHS_V = 50
checkpoint_filename_v = "visual_autoencoder.pth"

for param in visual_autoencoder.parameters():
    param.requires_grad = True

optimizer_v = torch.optim.AdamW(
    [
        {"params": visual_autoencoder.encoder.parameters(), "lr": 1e-5},
        {"params": visual_autoencoder.decoder.parameters(), "lr": 1e-4},
    ],
    weight_decay=1e-4
)

criterion = ReconstructionLoss().to(device)

# =========================================================
# SEQUENCE-READINESS REGULARISATION
# =========================================================
lambda_latent = 0.15      # latent stability
lambda_noise = 0.05       # robustness
lambda_skip = 0.02        # feature stability

early_stopper_v = EarlyStopping(patience=10, verbose=True)

# =========================================================
# LOAD CHECKPOINT
# =========================================================
try:
    visual_autoencoder, optimizer_v, start_epoch_v, initial_loss_v = load_checkpoint_from_drive(
        visual_autoencoder,
        optimizer_v,
        filename=checkpoint_filename_v
    )
    print(f"Resuming from epoch {start_epoch_v + 1}")
    early_stopper_v.best_loss = initial_loss_v
except FileNotFoundError:
    print("No checkpoint found, training from scratch.")

# =========================================================
# TRAINING LOOP
# =========================================================

global_step = 0

for epoch in range(start_epoch_v, N_EPOCHS_V):

    visual_autoencoder.train()
    epoch_loss = 0.0

    pbar = tqdm(
        autoencoder_dataloader,
        desc=f"Epoch {epoch+1}/{N_EPOCHS_V}"
    )

    for batch_idx, batch in enumerate(pbar):

        images = batch[0].to(device)
        images = F.interpolate(
            images,
            size=(224, 224),
            mode="bilinear",
            align_corners=False
        )

        optimizer_v.zero_grad()

        # FORWARD PASS
        z, mu, logvar, spatial = visual_autoencoder.encoder(images)
        recon = visual_autoencoder.decoder(z, spatial)

        losses = criterion(recon, images, mu, logvar)
        rec_loss = losses["total"]

        # LATENT STABILITY
        noise_img = images + torch.randn_like(images) * 0.03
        noise_img = torch.clamp(noise_img, 0, 1)

        z_noisy, _, _, _ = visual_autoencoder.encoder(noise_img)

        latent_consistency = F.mse_loss(z, z_noisy)

        # DECODER ROBUSTNESS
        z_perturbed = z + torch.randn_like(z) * 0.08
        recon_perturbed = visual_autoencoder.decoder(z_perturbed, spatial)

        robustness_loss = F.mse_loss(
            recon_perturbed,
            recon
        )

        # TOTAL LOSS
        loss = (
            rec_loss
            + lambda_latent * latent_consistency
            + lambda_noise * robustness_loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            visual_autoencoder.parameters(),
            1.0
        )

        optimizer_v.step()

        epoch_loss += loss.item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "recon": f"{losses['pixel']:.4f}",
            "latent": f"{latent_consistency:.4f}",
            "rob": f"{robustness_loss:.4f}"
        })

        if global_step % 50 == 0:
            with torch.no_grad():
                print(
                    f"\n[STEP {global_step}] "
                    f"Recon: {losses['pixel']:.6f} | "
                    f"Latent: {latent_consistency:.6f} | "
                    f"Rob: {robustness_loss:.6f}"
                )

        if global_step % 300 == 0:
            visual_autoencoder.eval()

            with torch.no_grad():

                fig, ax = plt.subplots(1, 2, figsize=(8, 4))

                ax[0].imshow(
                    images[0].cpu().permute(1,2,0).clamp(0,1)
                )
                ax[0].set_title("Original")

                ax[1].imshow(
                    recon[0].cpu().permute(1,2,0).clamp(0,1)
                )
                ax[1].set_title("Reconstruction")

                plt.show()

            visual_autoencoder.train()

        global_step += 1

    avg_train = epoch_loss / len(autoencoder_dataloader)

    print(
        f"\nEpoch {epoch+1}/{N_EPOCHS_V} "
        f"| Avg Loss: {avg_train:.6f}"
    )

    save_checkpoint_to_drive(
        visual_autoencoder,
        optimizer_v,
        epoch + 1,
        avg_train,
        filename=checkpoint_filename_v
    )

    if early_stopper_v.step(avg_train):
        break